In [6]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")


830

In [7]:
#Task 1 — Aggregation and Grouping

query = """
SELECT
    CustomerID,
    COUNT(*) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC;
"""

result_df = pd.read_sql_query(query, conn)

# Display Top 10 rows
print(result_df.head(10))

  customerID  order_count  total_freight  avg_freight
0      SAVEA           31        6683.70   215.603226
1      ERNSH           30        6205.39   206.846333
2      QUICK           28        5605.63   200.201071
3      HUNGO           19        2755.24   145.012632
4      RATTC           18        2134.21   118.567222
5      QUEEN           13        1982.70   152.515385
6      FOLKO           19        1678.08    88.320000
7      BERGS           18        1559.52    86.640000
8      FRANK           15        1403.44    93.562667
9      MEREP           13        1394.22   107.247692


In [8]:
#Task 2 — WHERE vs. HAVING

#Using WHERE

query_a = """
SELECT
    CustomerID,
    COUNT(*) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID;
"""

df_a = pd.read_sql_query(query_a, conn)
print(df_a.head(10))

#Using HAVING

query_b = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500;
"""

df_b = pd.read_sql_query(query_b, conn)
print(df_b.head(10))

#Query A focuses on qualifying individual orders, while Query B evaluates the overall aggregated value per customer.

  customerID  high_freight_orders
0      ALFKI                    2
1      ANTON                    2
2      AROUT                    2
3      BERGS                   11
4      BLAUS                    1
5      BLONP                    6
6      BOLID                    2
7      BONAP                   10
8      BOTTM                    6
9      BSBEV                    1
  customerID  total_freight
0      BERGS        1559.52
1      BLONP         623.66
2      BONAP        1357.87
3      BOTTM         793.95
4      EASTC         832.34
5      ERNSH        6205.39
6      FOLIG         637.94
7      FOLKO        1678.08
8      FRANK        1403.44
9      GODOS         568.27


In [9]:
#Task 3 — JOIN and Aggregation

#Using INNER

query_inner = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
    ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CompanyName, c.Country;
"""

df_inner = pd.read_sql_query(query_inner, conn)
print(df_inner.head(10))


#Using LEFT

query_left = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    COALESCE(SUM(o.Freight), 0) AS total_freight
FROM customers c
LEFT JOIN orders o
    ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CompanyName, c.Country;
"""

df_left = pd.read_sql_query(query_left, conn)
print(df_left.head(10))


#The INNER JOIN returns only customers who have matching rows in the orders table, so customers with no orders are excluded. The LEFT JOIN includes all customers regardless of whether they have orders, filling missing order data with NULL

                          companyName  country  order_count  total_freight
0                 Alfreds Futterkiste  Germany            6         225.58
1  Ana Trujillo Emparedados y helados   Mexico            4          97.42
2             Antonio Moreno Taquería   Mexico            7         268.52
3                     Around the Horn       UK           13         471.95
4                  Berglunds snabbköp   Sweden           18        1559.52
5             Blauer See Delikatessen  Germany            7         168.26
6            Blondesddsl père et fils   France           11         623.66
7           Bólido Comidas preparadas    Spain            3         191.17
8                            Bon app'   France           17        1357.87
9               Bottom-Dollar Markets   Canada           14         793.95
                          companyName  country  order_count  total_freight
0                 Alfreds Futterkiste  Germany            6         225.58
1  Ana Trujillo Emparedad